# 03 — Representasi Teks: N-Gram, TF, IDF, TF-IDF, Hyperparameter, dan Similarity

Notebook ini merupakan lanjutan langsung dari **Notebook 02 — Text Preprocessing**.

Pada Notebook 02, teks mentah dibersihkan melalui tahapan seperti cleaning, case folding, tokenization, stopword handling, slang normalization, negation handling, dan stemming/lemmatization. Pada notebook ini, kita masuk ke pertanyaan berikutnya:

> Setelah teks bersih diperoleh, bagaimana teks tersebut diubah menjadi representasi numerik yang dapat diproses oleh algoritma machine learning?

Notebook ini membahas secara bertahap:

1. Mengapa teks harus diubah menjadi vektor.
2. Vocabulary dan document-term matrix.
3. Unigram, bigram, trigram, dan n-gram.
4. Trade-off n-gram: konteks lokal vs sparsity.
5. Term Frequency (TF) dan variasinya.
6. Document Frequency (DF) dan Inverse Document Frequency (IDF).
7. Rumus TF-IDF klasik dan versi smoothed IDF seperti pada scikit-learn.
8. Perhitungan manual TF-IDF langkah demi langkah.
9. Normalisasi vektor: L1, L2, dan tanpa normalisasi.
10. Cosine similarity.
11. Hyperparameter penting `TfidfVectorizer`.
12. Eksperimen perubahan hyperparameter dan dampaknya.
13. Studi kasus information retrieval sederhana.
14. Latihan mahasiswa.

**Catatan penting:** notebook ini sengaja dibuat pedagogis. Beberapa bagian menghitung manual agar mahasiswa memahami logika matematisnya sebelum menggunakan library.

## 1. Posisi Notebook 03 dalam Pipeline NLP

Pipeline klasik NLP dapat disederhanakan sebagai berikut:

```text
Raw Text
→ Preprocessing
→ Token Bersih
→ Representasi Numerik
→ Model / Similarity / Retrieval / Clustering / Classification
```

Notebook 02 berhenti pada tahap **token bersih**. Notebook 03 memulai tahap **representasi numerik**.

Algoritma machine learning tidak memahami teks secara langsung. Model seperti Logistic Regression, SVM, Naive Bayes, K-Means, atau cosine similarity bekerja pada angka. Oleh karena itu, teks perlu diubah menjadi vektor.

Contoh sederhana:

```text
"analisis data penting"
```

dapat direpresentasikan sebagai vektor frekuensi:

```text
[data=1, analisis=1, penting=1, sistem=0, informasi=0]
```

Representasi ini disebut salah satu bentuk **Vector Space Model (VSM)**.

In [1]:
import math
import re
from collections import Counter, defaultdict
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)

def print_section(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def print_subsection(title: str):
    print("\n" + "-" * 100)
    print(title)
    print("-" * 100)

def tokenize(text: str) -> List[str]:
    """Tokenisasi sederhana berbasis regex.
    Cocok untuk contoh kelas, bukan tokenizer produksi.
    """
    return re.findall(r"(?u)\b\w+\b", text.lower())

def show_df(df: pd.DataFrame, title: str | None = None):
    if title:
        print_section(title)
    display(df)

## 2. Corpus Contoh yang Dipakai Sepanjang Notebook

Agar pembahasan konsisten, kita gunakan corpus kecil berbahasa Indonesia. Corpus ini sengaja memuat beberapa tema:

- analisis data,
- machine learning,
- NLP,
- sistem informasi,
- kualitas data.

Dengan corpus kecil, perhitungan manual masih mudah diikuti. Prinsip yang sama berlaku untuk corpus besar.

In [2]:
corpus = [
    "analisis data penting untuk sistem informasi",
    "data mining dan machine learning untuk analisis bisnis",
    "bahasa alami digunakan dalam sistem NLP modern",
    "sistem informasi membantu pengambilan keputusan",
    "machine learning membutuhkan data berkualitas",
]

doc_ids = [f"D{i+1}" for i in range(len(corpus))]
corpus_tokens = [tokenize(doc) for doc in corpus]

for doc_id, raw, toks in zip(doc_ids, corpus, corpus_tokens):
    print(f"{doc_id}: {raw}")
    print(f"    tokens = {toks}")

D1: analisis data penting untuk sistem informasi
    tokens = ['analisis', 'data', 'penting', 'untuk', 'sistem', 'informasi']
D2: data mining dan machine learning untuk analisis bisnis
    tokens = ['data', 'mining', 'dan', 'machine', 'learning', 'untuk', 'analisis', 'bisnis']
D3: bahasa alami digunakan dalam sistem NLP modern
    tokens = ['bahasa', 'alami', 'digunakan', 'dalam', 'sistem', 'nlp', 'modern']
D4: sistem informasi membantu pengambilan keputusan
    tokens = ['sistem', 'informasi', 'membantu', 'pengambilan', 'keputusan']
D5: machine learning membutuhkan data berkualitas
    tokens = ['machine', 'learning', 'membutuhkan', 'data', 'berkualitas']


## 3. Vocabulary dan Document-Term Matrix

### 3.1 Vocabulary

**Vocabulary** adalah himpunan term unik yang diperoleh dari corpus.

Jika corpus terdiri dari beberapa dokumen, maka semua token unik dari seluruh dokumen dikumpulkan menjadi daftar fitur.

Contoh:

```text
D1 = analisis data penting
D2 = data mining
```

Vocabulary:

```text
[analisis, data, mining, penting]
```

### 3.2 Document-Term Matrix

Setelah vocabulary dibentuk, setiap dokumen dapat diubah menjadi baris vektor. Kolomnya adalah term dalam vocabulary.

Nilai pada matriks dapat berupa:

- jumlah kemunculan term,
- binary presence/absence,
- TF,
- TF-IDF,
- atau bobot lain.

Matriks inilah yang menjadi input banyak algoritma NLP klasik.

In [3]:
def build_vocab(tokenized_docs: List[List[str]]) -> List[str]:
    return sorted(set(token for doc in tokenized_docs for token in doc))

vocab = build_vocab(corpus_tokens)
print("Vocabulary:")
print(vocab)
print("\nJumlah fitur:", len(vocab))

count_matrix = []
for tokens in corpus_tokens:
    counts = Counter(tokens)
    count_matrix.append([counts.get(term, 0) for term in vocab])

count_df = pd.DataFrame(count_matrix, columns=vocab, index=doc_ids)
show_df(count_df, "Document-Term Matrix Berbasis Raw Count")

Vocabulary:
['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']

Jumlah fitur: 22

Document-Term Matrix Berbasis Raw Count


,alami,analisis,bahasa,berkualitas,bisnis,dalam,dan,data,digunakan,informasi,keputusan,learning,machine,membantu,membutuhkan,mining,modern,nlp,pengambilan,penting,sistem,untuk
D1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,1
D2,0,1,0,0,1,0,1,1,0,0,0,1,1,0,0,1,0,0,0,0,0,1
D3,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0
D4,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,1,0,1,0
D5,0,0,0,1,0,0,0,1,0,0,0,1,1,0,1,0,0,0,0,0,0,0


## 4. N-Gram: Unigram, Bigram, Trigram, dan Generalisasi N-Gram

### 4.1 Definisi

N-gram adalah urutan token sepanjang `n`.

- **Unigram**: satu token.
- **Bigram**: dua token berurutan.
- **Trigram**: tiga token berurutan.
- **N-gram**: urutan `n` token berurutan.

Contoh kalimat:

```text
analisis data penting
```

Unigram:

```text
analisis, data, penting
```

Bigram:

```text
analisis data, data penting
```

Trigram:

```text
analisis data penting
```

### 4.2 Mengapa n-gram penting?

Unigram hanya menangkap kata secara individual. Bigram dan trigram menangkap konteks lokal.

Contoh:

```text
machine learning
```

Jika hanya memakai unigram, kata `machine` dan `learning` diperlakukan terpisah. Dengan bigram, frasa `machine learning` menjadi fitur tersendiri.

### 4.3 Trade-off

Semakin besar `n`, semakin kaya konteks yang ditangkap, tetapi:

1. jumlah fitur meningkat,
2. matriks menjadi lebih sparse,
3. risiko overfitting meningkat pada dataset kecil,
4. kebutuhan memori meningkat.

In [4]:
def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def stringify_ngrams(grams: List[Tuple[str, ...]]) -> List[str]:
    return [" ".join(g) for g in grams]

example = tokenize("analisis data penting untuk sistem informasi")
print("Tokens:", example)
for n in [1, 2, 3, 4]:
    grams = stringify_ngrams(make_ngrams(example, n))
    print(f"{n}-gram:", grams)

Tokens: ['analisis', 'data', 'penting', 'untuk', 'sistem', 'informasi']
1-gram: ['analisis', 'data', 'penting', 'untuk', 'sistem', 'informasi']
2-gram: ['analisis data', 'data penting', 'penting untuk', 'untuk sistem', 'sistem informasi']
3-gram: ['analisis data penting', 'data penting untuk', 'penting untuk sistem', 'untuk sistem informasi']
4-gram: ['analisis data penting untuk', 'data penting untuk sistem', 'penting untuk sistem informasi']


In [5]:
def vocab_for_ngram_range(corpus: List[str], ngram_range: Tuple[int, int]) -> List[str]:
    min_n, max_n = ngram_range
    vocab_ngram = set()
    for doc in corpus:
        toks = tokenize(doc)
        for n in range(min_n, max_n + 1):
            vocab_ngram.update(stringify_ngrams(make_ngrams(toks, n)))
    return sorted(vocab_ngram)

rows = []
for ngram_range in [(1,1), (1,2), (1,3), (2,2), (2,3)]:
    features = vocab_for_ngram_range(corpus, ngram_range)
    rows.append({
        "ngram_range": str(ngram_range),
        "jumlah_fitur": len(features),
        "contoh_fitur_pertama": ", ".join(features[:10])
    })

ngram_growth_df = pd.DataFrame(rows)
show_df(ngram_growth_df, "Pertumbuhan Jumlah Fitur akibat ngram_range")


Pertumbuhan Jumlah Fitur akibat ngram_range


,ngram_range,jumlah_fitur,contoh_fitur_pertama
0,"(1, 1)",22,"alami, analisis, bahasa, berkualitas, bisnis, ..."
1,"(1, 2)",46,"alami, alami digunakan, analisis, analisis bis..."
2,"(1, 3)",67,"alami, alami digunakan, alami digunakan dalam,..."
3,"(2, 2)",24,"alami digunakan, analisis bisnis, analisis dat..."
4,"(2, 3)",45,"alami digunakan, alami digunakan dalam, analis..."


## 5. Sparsity pada N-Gram

**Sparsity** berarti sebagian besar nilai matriks adalah nol.

N-gram yang lebih panjang biasanya lebih jarang muncul. Pada corpus kecil, trigram sering hanya muncul sekali. Ini membuat fitur menjadi sangat spesifik terhadap dokumen tertentu.

Sparsity bukan selalu buruk. Dalam information retrieval, frasa spesifik bisa sangat berguna. Namun dalam klasifikasi, fitur yang terlalu jarang bisa membuat model sulit melakukan generalisasi.

In [6]:
def ngram_document_frequency(corpus: List[str], n: int) -> pd.DataFrame:
    df_counter = Counter()
    for doc in corpus:
        grams = set(stringify_ngrams(make_ngrams(tokenize(doc), n)))
        df_counter.update(grams)
    return pd.DataFrame(
        sorted(df_counter.items(), key=lambda x: (-x[1], x[0])),
        columns=["ngram", "document_frequency"]
    )

for n in [1, 2, 3]:
    show_df(ngram_document_frequency(corpus, n), f"Document Frequency untuk {n}-gram")


Document Frequency untuk 1-gram


,ngram,document_frequency
0,data,3
1,sistem,3
2,analisis,2
3,informasi,2
4,learning,2
5,machine,2
6,untuk,2
7,alami,1
8,bahasa,1
9,berkualitas,1



Document Frequency untuk 2-gram


,ngram,document_frequency
0,machine learning,2
1,sistem informasi,2
2,alami digunakan,1
3,analisis bisnis,1
4,analisis data,1
5,bahasa alami,1
6,dalam sistem,1
7,dan machine,1
8,data berkualitas,1
9,data mining,1



Document Frequency untuk 3-gram


,ngram,document_frequency
0,alami digunakan dalam,1
1,analisis data penting,1
2,bahasa alami digunakan,1
3,dalam sistem nlp,1
4,dan machine learning,1
5,data mining dan,1
6,data penting untuk,1
7,digunakan dalam sistem,1
8,informasi membantu pengambilan,1
9,learning membutuhkan data,1


## 6. Term Frequency (TF)

TF mengukur seberapa sering sebuah term muncul dalam sebuah dokumen.

Namun, ada banyak variasi TF. Pilihan variasi TF memengaruhi skala bobot dan interpretasi fitur.

Misalkan:

- `f(t, d)` = jumlah kemunculan term `t` pada dokumen `d`.
- `|d|` = jumlah token dalam dokumen `d`.
- `max_f(d)` = frekuensi term tertinggi dalam dokumen `d`.

### 6.1 Raw TF

\[
TF_{raw}(t,d) = f(t,d)
\]

Kelebihan:

- mudah dipahami,
- cocok untuk dokumen dengan panjang relatif seragam.

Kekurangan:

- bias terhadap dokumen panjang.

### 6.2 Binary TF

\[
TF_{binary}(t,d) =
\begin{cases}
1, & \text{jika } f(t,d) > 0 \\
0, & \text{jika } f(t,d) = 0
\end{cases}
\]

Binary TF hanya peduli apakah term muncul atau tidak.

### 6.3 Normalized TF

\[
TF_{norm}(t,d) = \frac{f(t,d)}{|d|}
\]

Normalized TF mengurangi bias panjang dokumen.

### 6.4 Log TF / Sublinear TF

\[
TF_{log}(t,d) =
\begin{cases}
1 + \log(f(t,d)), & \text{jika } f(t,d) > 0 \\
0, & \text{jika } f(t,d) = 0
\end{cases}
\]

Log TF menekan dominasi term yang muncul terlalu sering.

### 6.5 Augmented TF

\[
TF_{aug}(t,d) = 0.5 + 0.5 \times \frac{f(t,d)}{max_f(d)}
\]

Augmented TF membandingkan frekuensi term dengan term paling sering di dokumen tersebut.

In [7]:
def raw_tf(term: str, tokens: List[str]) -> int:
    return tokens.count(term)

def binary_tf(term: str, tokens: List[str]) -> int:
    return int(term in tokens)

def normalized_tf(term: str, tokens: List[str]) -> float:
    return tokens.count(term) / len(tokens) if tokens else 0.0

def log_tf(term: str, tokens: List[str]) -> float:
    f = tokens.count(term)
    return 0.0 if f == 0 else 1 + math.log(f)

def augmented_tf(term: str, tokens: List[str]) -> float:
    if not tokens:
        return 0.0
    counts = Counter(tokens)
    f = counts.get(term, 0)
    if f == 0:
        return 0.0
    return 0.5 + 0.5 * (f / max(counts.values()))

def tf_variants(term: str, tokens: List[str]) -> Dict[str, float]:
    return {
        "raw_tf": raw_tf(term, tokens),
        "binary_tf": binary_tf(term, tokens),
        "normalized_tf": normalized_tf(term, tokens),
        "log_tf": log_tf(term, tokens),
        "augmented_tf": augmented_tf(term, tokens),
    }

example_doc = tokenize("data data data analisis sistem penting")
terms_to_check = ["data", "analisis", "machine"]
rows = []
for term in terms_to_check:
    row = {"term": term}
    row.update(tf_variants(term, example_doc))
    rows.append(row)

tf_df = pd.DataFrame(rows)
show_df(tf_df, "Perbandingan Varian TF pada Satu Dokumen")


Perbandingan Varian TF pada Satu Dokumen


,term,raw_tf,binary_tf,normalized_tf,log_tf,augmented_tf
0,data,3,1,0.5000,2.0986,1.0000
1,analisis,1,1,0.1667,1.0000,0.6667
2,machine,0,0,0.0000,0.0000,0.0000


## 7. Dampak Varian TF terhadap Dokumen Panjang dan Pendek

Raw TF bisa menyesatkan ketika membandingkan dokumen dengan panjang berbeda.

Contoh:

```text
D pendek: data data analisis
D panjang: data data analisis sistem informasi data besar untuk analisis bisnis modern
```

Raw TF untuk `data` pada dokumen panjang bisa lebih besar, tetapi secara proporsional belum tentu lebih dominan.

In [8]:
doc_short = tokenize("data data analisis")
doc_long = tokenize("data data analisis sistem informasi data besar untuk analisis bisnis modern")

rows = []
for name, tokens in [("pendek", doc_short), ("panjang", doc_long)]:
    row = {"dokumen": name, "jumlah_token": len(tokens)}
    row.update(tf_variants("data", tokens))
    rows.append(row)

show_df(pd.DataFrame(rows), "Dampak Varian TF pada Dokumen Panjang vs Pendek")


Dampak Varian TF pada Dokumen Panjang vs Pendek


,dokumen,jumlah_token,raw_tf,binary_tf,normalized_tf,log_tf,augmented_tf
0,pendek,3,2,1,0.6667,1.6931,1.0
1,panjang,11,3,1,0.2727,2.0986,1.0


## 8. Document Frequency (DF)

Document Frequency menghitung jumlah dokumen yang mengandung suatu term.

\[
DF(t) = |\{d \in D : t \in d\}|
\]

Dengan kata lain, DF tidak menghitung berapa kali term muncul di seluruh corpus. DF menghitung **berapa dokumen** yang mengandung term tersebut.

Contoh:

Jika term `data` muncul 3 kali di D1 dan 1 kali di D5, maka:

```text
DF(data) = 2
```

bukan 4.

DF penting karena menjadi dasar IDF.

In [9]:
def document_frequency(term: str, tokenized_docs: List[List[str]]) -> int:
    return sum(1 for tokens in tokenized_docs if term in set(tokens))

df_rows = []
for term in vocab:
    df_rows.append({
        "term": term,
        "document_frequency": document_frequency(term, corpus_tokens),
        "corpus_frequency": sum(tokens.count(term) for tokens in corpus_tokens)
    })

df_table = pd.DataFrame(df_rows).sort_values(["document_frequency", "term"], ascending=[False, True])
show_df(df_table, "Document Frequency vs Corpus Frequency")


Document Frequency vs Corpus Frequency


,term,document_frequency,corpus_frequency
7,data,3,3
20,sistem,3,3
1,analisis,2,2
9,informasi,2,2
11,learning,2,2
12,machine,2,2
21,untuk,2,2
0,alami,1,1
2,bahasa,1,1
3,berkualitas,1,1


## 9. Inverse Document Frequency (IDF)

IDF mengukur seberapa spesifik sebuah term dalam corpus.

Intuisi IDF:

- Term yang muncul di hampir semua dokumen dianggap kurang diskriminatif.
- Term yang muncul di sedikit dokumen dianggap lebih spesifik.

### 9.1 IDF Klasik

\[
IDF_{classic}(t) = \log \left(\frac{N}{DF(t)}\right)
\]

Dengan:

- `N` = jumlah dokumen dalam corpus,
- `DF(t)` = jumlah dokumen yang mengandung term `t`.

Jika `DF(t) = N`, maka:

\[
IDF(t) = \log(1) = 0
\]

Artinya term tersebut tidak membantu membedakan dokumen.

### 9.2 Smoothed IDF versi scikit-learn

Saat `smooth_idf=True`, scikit-learn memakai formula:

\[
IDF_{smooth}(t) = \log \left(\frac{1 + N}{1 + DF(t)}\right) + 1
\]

Smoothing membuat nilai IDF lebih stabil, terutama pada corpus kecil.

### 9.3 Mengapa ada `+1`?

Penambahan `+1` menjaga agar bobot IDF tidak menjadi nol untuk term yang muncul di semua dokumen. Ini membuat term umum tetap memiliki bobot kecil, bukan sepenuhnya hilang.

In [10]:
def idf_classic(N: int, df: int) -> float:
    return 0.0 if df == 0 else math.log(N / df)

def idf_smooth_sklearn(N: int, df: int) -> float:
    return math.log((1 + N) / (1 + df)) + 1

N = len(corpus_tokens)
idf_rows = []
for term in vocab:
    df = document_frequency(term, corpus_tokens)
    idf_rows.append({
        "term": term,
        "DF": df,
        "IDF_classic_log(N/DF)": idf_classic(N, df),
        "IDF_smooth_sklearn": idf_smooth_sklearn(N, df),
    })

idf_df = pd.DataFrame(idf_rows).sort_values("IDF_classic_log(N/DF)", ascending=False)
show_df(idf_df, "Perbandingan IDF Klasik vs Smoothed IDF scikit-learn")


Perbandingan IDF Klasik vs Smoothed IDF scikit-learn


,term,DF,IDF_classic_log(N/DF),IDF_smooth_sklearn
0,alami,1,1.6094,2.0986
2,bahasa,1,1.6094,2.0986
4,bisnis,1,1.6094,2.0986
3,berkualitas,1,1.6094,2.0986
5,dalam,1,1.6094,2.0986
6,dan,1,1.6094,2.0986
18,pengambilan,1,1.6094,2.0986
8,digunakan,1,1.6094,2.0986
13,membantu,1,1.6094,2.0986
10,keputusan,1,1.6094,2.0986


## 10. TF-IDF: Formula Dasar

TF-IDF menggabungkan dua ide:

1. Term penting jika sering muncul dalam dokumen tertentu.
2. Term lebih bermakna jika tidak muncul di terlalu banyak dokumen.

Formula umum:

\[
TFIDF(t,d) = TF(t,d) \times IDF(t)
\]

Jika memakai raw TF dan IDF klasik:

\[
TFIDF(t,d) = f(t,d) \times \log \left(\frac{N}{DF(t)}\right)
\]

Jika memakai normalized TF dan IDF klasik:

\[
TFIDF(t,d) = \frac{f(t,d)}{|d|} \times \log \left(\frac{N}{DF(t)}\right)
\]

Jika memakai log TF dan smoothed IDF:

\[
TFIDF(t,d) = (1 + \log(f(t,d))) \times \left[\log \left(\frac{1+N}{1+DF(t)}\right)+1\right]
\]

untuk `f(t,d) > 0`.

### Catatan penting

Tidak ada satu formula TF-IDF yang selalu paling benar. Formula yang dipakai bergantung pada library, parameter, dan tujuan analisis.

In [11]:
def tfidf(tf_value: float, idf_value: float) -> float:
    return tf_value * idf_value

def compute_tfidf_table(tokens: List[str], tokenized_docs: List[List[str]], vocab: List[str], tf_func, idf_func) -> pd.DataFrame:
    N = len(tokenized_docs)
    rows = []
    for term in vocab:
        df = document_frequency(term, tokenized_docs)
        tf_val = tf_func(term, tokens)
        idf_val = idf_func(N, df)
        rows.append({
            "term": term,
            "TF": tf_val,
            "DF": df,
            "IDF": idf_val,
            "TF-IDF": tf_val * idf_val,
        })
    return pd.DataFrame(rows).sort_values("TF-IDF", ascending=False)

d1_tokens = corpus_tokens[0]
manual_tfidf_d1 = compute_tfidf_table(d1_tokens, corpus_tokens, vocab, raw_tf, idf_classic)
show_df(manual_tfidf_d1, "TF-IDF Manual D1: Raw TF × IDF Klasik")


TF-IDF Manual D1: Raw TF × IDF Klasik


,term,TF,DF,IDF,TF-IDF
19,penting,1,1,1.6094,1.6094
1,analisis,1,2,0.9163,0.9163
9,informasi,1,2,0.9163,0.9163
21,untuk,1,2,0.9163,0.9163
7,data,1,3,0.5108,0.5108
20,sistem,1,3,0.5108,0.5108
5,dalam,0,1,1.6094,0.0000
4,bisnis,0,1,1.6094,0.0000
3,berkualitas,0,1,1.6094,0.0000
2,bahasa,0,1,1.6094,0.0000


## 11. Studi Kasus Manual: Menghitung TF-IDF Term `data` pada D1

Ambil dokumen:

```text
D1 = analisis data penting untuk sistem informasi
```

Term yang dihitung:

```text
data
```

Langkah:

1. Hitung TF pada D1.
2. Hitung DF pada corpus.
3. Hitung IDF.
4. Kalikan TF dan IDF.

Jika `data` muncul 1 kali di D1:

\[
TF(data, D1) = 1
\]

Jika `data` muncul pada 3 dokumen dari 5:

\[
DF(data) = 3, \quad N = 5
\]

IDF klasik:

\[
IDF(data) = \log(5/3)
\]

Maka:

\[
TFIDF(data, D1) = 1 \times \log(5/3)
\]

In [12]:
term = "data"
doc_index = 0
tokens = corpus_tokens[doc_index]
N = len(corpus_tokens)
df = document_frequency(term, corpus_tokens)
tf_raw = raw_tf(term, tokens)
idf_c = idf_classic(N, df)
idf_s = idf_smooth_sklearn(N, df)

print(f"Dokumen: D{doc_index+1}")
print(f"Tokens: {tokens}")
print(f"Term: {term}")
print(f"TF raw = {tf_raw}")
print(f"N = {N}")
print(f"DF({term}) = {df}")
print(f"IDF klasik = log({N}/{df}) = {idf_c:.6f}")
print(f"TF-IDF klasik = {tf_raw} × {idf_c:.6f} = {tf_raw * idf_c:.6f}")
print()
print(f"IDF smoothed sklearn = log((1+{N})/(1+{df})) + 1 = {idf_s:.6f}")
print(f"TF-IDF smoothed = {tf_raw} × {idf_s:.6f} = {tf_raw * idf_s:.6f}")

Dokumen: D1
Tokens: ['analisis', 'data', 'penting', 'untuk', 'sistem', 'informasi']
Term: data
TF raw = 1
N = 5
DF(data) = 3
IDF klasik = log(5/3) = 0.510826
TF-IDF klasik = 1 × 0.510826 = 0.510826

IDF smoothed sklearn = log((1+5)/(1+3)) + 1 = 1.405465
TF-IDF smoothed = 1 × 1.405465 = 1.405465


## 12. Perbandingan TF Variant di dalam TF-IDF

Sekarang kita lihat dampak pilihan TF terhadap bobot TF-IDF.

Term yang sama dapat memiliki bobot akhir berbeda karena TF yang dipakai berbeda.

Ini penting karena parameter seperti `binary=True`, `sublinear_tf=True`, atau normalisasi panjang dokumen mengubah perilaku representasi.

In [13]:
term = "data"
tokens = tokenize("data data data analisis sistem penting")
N = len(corpus_tokens)
df = document_frequency(term, corpus_tokens)
idf_val = idf_smooth_sklearn(N, df)

rows = []
for name, func in [
    ("raw_tf", raw_tf),
    ("binary_tf", binary_tf),
    ("normalized_tf", normalized_tf),
    ("log_tf", log_tf),
    ("augmented_tf", augmented_tf),
]:
    tf_val = func(term, tokens)
    rows.append({
        "tf_variant": name,
        "TF": tf_val,
        "IDF_smooth": idf_val,
        "TF-IDF": tf_val * idf_val,
        "interpretasi": {
            "raw_tf": "mengikuti jumlah kemunculan mentah",
            "binary_tf": "hanya presence/absence",
            "normalized_tf": "dibagi panjang dokumen",
            "log_tf": "frekuensi tinggi ditekan",
            "augmented_tf": "dinormalisasi terhadap term paling sering",
        }[name]
    })

show_df(pd.DataFrame(rows), "Dampak Varian TF terhadap Bobot TF-IDF")


Dampak Varian TF terhadap Bobot TF-IDF


,tf_variant,TF,IDF_smooth,TF-IDF,interpretasi
0,raw_tf,3.0000,1.4055,4.2164,mengikuti jumlah kemunculan mentah
1,binary_tf,1.0000,1.4055,1.4055,hanya presence/absence
2,normalized_tf,0.5000,1.4055,0.7027,dibagi panjang dokumen
3,log_tf,2.0986,1.4055,2.9495,frekuensi tinggi ditekan
4,augmented_tf,1.0000,1.4055,1.4055,dinormalisasi terhadap term paling sering


## 13. Normalisasi Vektor: None, L1, dan L2

Setelah TF-IDF dihitung, vektor dokumen sering dinormalisasi.

### 13.1 Tanpa normalisasi

Vektor mempertahankan skala asli. Dokumen panjang bisa memiliki norma lebih besar.

### 13.2 L1 Normalization

\[
\|x\|_1 = \sum_i |x_i|
\]

Setelah normalisasi L1, jumlah nilai absolut komponen vektor menjadi 1.

### 13.3 L2 Normalization

\[
\|x\|_2 = \sqrt{\sum_i x_i^2}
\]

Setelah normalisasi L2, panjang Euclidean vektor menjadi 1.

L2 sangat umum digunakan untuk cosine similarity karena cosine similarity membandingkan arah vektor.

In [14]:
def l1_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.sum(np.abs(vec))
    return vec if norm == 0 else vec / norm

def l2_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.sqrt(np.sum(vec ** 2))
    return vec if norm == 0 else vec / norm

vec = np.array([3.0, 4.0, 0.0])
comparison = pd.DataFrame({
    "komponen": ["x1", "x2", "x3"],
    "original": vec,
    "L1_normalized": l1_normalize(vec),
    "L2_normalized": l2_normalize(vec),
})
show_df(comparison, "Contoh Normalisasi Vektor")

print("L1 norm original:", np.sum(np.abs(vec)))
print("L2 norm original:", np.sqrt(np.sum(vec ** 2)))
print("L1 norm setelah L1:", np.sum(np.abs(l1_normalize(vec))))
print("L2 norm setelah L2:", np.sqrt(np.sum(l2_normalize(vec) ** 2)))


Contoh Normalisasi Vektor


,komponen,original,L1_normalized,L2_normalized
0,x1,3.0,0.4286,0.6
1,x2,4.0,0.5714,0.8
2,x3,0.0,0.0000,0.0


L1 norm original: 7.0
L2 norm original: 5.0
L1 norm setelah L1: 1.0
L2 norm setelah L2: 1.0


## 14. Cosine Similarity

Cosine similarity mengukur kemiripan arah dua vektor.

\[
cosine(A,B) = \frac{A \cdot B}{\|A\|\|B\|}
\]

Nilai cosine similarity:

- mendekati 1: sangat mirip,
- mendekati 0: tidak mirip atau tidak berbagi fitur,
- bisa negatif jika menggunakan representasi yang memungkinkan nilai negatif, tetapi TF-IDF biasanya non-negatif.

Dalam TF-IDF, cosine similarity sering dipakai untuk:

1. information retrieval,
2. document similarity,
3. duplicate detection,
4. clustering,
5. rekomendasi dokumen.

In [15]:
vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", norm="l2")
X = vectorizer.fit_transform(corpus)
features = vectorizer.get_feature_names_out()

sim = cosine_similarity(X)
sim_df = pd.DataFrame(sim, index=doc_ids, columns=doc_ids)
show_df(sim_df, "Cosine Similarity antar Dokumen menggunakan TF-IDF L2")

print("Dokumen paling mirip untuk setiap dokumen:")
for i, doc_id in enumerate(doc_ids):
    scores = sim[i].copy()
    scores[i] = -1
    j = int(np.argmax(scores))
    print(f"{doc_id} paling mirip dengan {doc_ids[j]} | score={scores[j]:.4f}")


Cosine Similarity antar Dokumen menggunakan TF-IDF L2


,D1,D2,D3,D4,D5
D1,1.0000,0.3626,0.0900,0.2768,0.1180
D2,0.3626,1.0000,0.0000,0.0000,0.3674
D3,0.0900,0.0000,1.0000,0.0872,0.0000
D4,0.2768,0.0000,0.0872,1.0000,0.0000
D5,0.1180,0.3674,0.0000,0.0000,1.0000


Dokumen paling mirip untuk setiap dokumen:
D1 paling mirip dengan D2 | score=0.3626
D2 paling mirip dengan D5 | score=0.3674
D3 paling mirip dengan D1 | score=0.0900
D4 paling mirip dengan D1 | score=0.2768
D5 paling mirip dengan D2 | score=0.3674


## 15. TfidfVectorizer: Parameter dan Hyperparameter Penting

Dalam scikit-learn, `TfidfVectorizer` menggabungkan beberapa tahap:

```text
raw text → tokenization → vocabulary → count matrix → TF-IDF weighting → normalization
```

Parameter penting:

| Parameter | Fungsi | Dampak jika diubah |
|---|---|---|
| `lowercase` | Mengubah teks menjadi huruf kecil | Mengurangi fitur duplikat seperti `Data` dan `data` |
| `token_pattern` | Regex untuk menentukan token valid | Memengaruhi apakah angka, satu huruf, atau simbol masuk sebagai token |
| `ngram_range` | Rentang n-gram | Makin besar konteks, makin banyak fitur dan makin sparse |
| `stop_words` | Stopword list | Menghapus kata umum, tetapi berisiko menghapus negasi |
| `min_df` | Minimum document frequency | Membuang term terlalu jarang |
| `max_df` | Maximum document frequency | Membuang term terlalu umum |
| `max_features` | Batas jumlah fitur | Menghemat memori, tetapi bisa membuang informasi |
| `binary` | Count dibuat 0/1 | Cocok jika kehadiran term lebih penting daripada frekuensi |
| `use_idf` | Aktifkan IDF | Jika False, bobot hanya berbasis TF |
| `smooth_idf` | Smoothing IDF | Menstabilkan IDF pada corpus kecil |
| `sublinear_tf` | Gunakan `1 + log(tf)` | Menekan dominasi term berulang |
| `norm` | Normalisasi vektor | Memengaruhi similarity dan skala fitur |

Bagian berikutnya menunjukkan eksperimen nyata dari perubahan parameter tersebut.

In [16]:
def vectorizer_report(vectorizer, corpus: List[str], title: str, top_n_features: int = 30):
    X = vectorizer.fit_transform(corpus)
    features = vectorizer.get_feature_names_out()
    dense = X.toarray()
    df = pd.DataFrame(dense, columns=features, index=doc_ids)
    print_section(title)
    print("Shape matriks:", X.shape)
    print("Jumlah fitur:", len(features))
    print("Contoh fitur:", list(features[:top_n_features]))
    display(df)
    return X, features, df

## 16. Eksperimen 1 — Dampak `ngram_range`

`ngram_range` menentukan fitur berbasis urutan token.

- `(1,1)` hanya unigram.
- `(1,2)` unigram + bigram.
- `(1,3)` unigram + bigram + trigram.

### Dampak konseptual

Jika `ngram_range` dinaikkan:

1. konteks lokal lebih tertangkap,
2. frasa seperti `machine learning` menjadi fitur,
3. jumlah fitur meningkat,
4. matriks makin sparse,
5. risiko overfitting meningkat pada dataset kecil.

In [17]:
for ngr in [(1,1), (1,2), (1,3)]:
    vec = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=ngr,
        use_idf=True,
        smooth_idf=True,
        norm="l2"
    )
    X_tmp = vec.fit_transform(corpus)
    features_tmp = vec.get_feature_names_out()
    nonzero_ratio = X_tmp.count_nonzero() / (X_tmp.shape[0] * X_tmp.shape[1])
    print(f"ngram_range={ngr} | shape={X_tmp.shape} | fitur={len(features_tmp)} | nonzero_ratio={nonzero_ratio:.4f}")
    print("contoh fitur:", list(features_tmp[:15]))
    print()

ngram_range=(1, 1) | shape=(5, 22) | fitur=22 | nonzero_ratio=0.2818
contoh fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan']

ngram_range=(1, 2) | shape=(5, 46) | fitur=46 | nonzero_ratio=0.2478
contoh fitur: ['alami', 'alami digunakan', 'analisis', 'analisis bisnis', 'analisis data', 'bahasa', 'bahasa alami', 'berkualitas', 'bisnis', 'dalam', 'dalam sistem', 'dan', 'dan machine', 'data', 'data berkualitas']

ngram_range=(1, 3) | shape=(5, 67) | fitur=67 | nonzero_ratio=0.2328
contoh fitur: ['alami', 'alami digunakan', 'alami digunakan dalam', 'analisis', 'analisis bisnis', 'analisis data', 'analisis data penting', 'bahasa', 'bahasa alami', 'bahasa alami digunakan', 'berkualitas', 'bisnis', 'dalam', 'dalam sistem', 'dalam sistem nlp']



## 17. Eksperimen 2 — Dampak `min_df`

`min_df` membuang term yang muncul di terlalu sedikit dokumen.

Contoh:

```python
min_df=2
```

berarti term harus muncul minimal di 2 dokumen.

### Dampak konseptual

Jika `min_df` dinaikkan:

1. fitur langka dibuang,
2. noise berkurang,
3. dimensi matriks turun,
4. memori lebih hemat,
5. tetapi term penting yang memang spesifik bisa ikut hilang.

Untuk dataset kecil, `min_df=2` bisa terlalu agresif. Untuk dataset besar, `min_df=2`, `min_df=5`, atau `min_df=10` sering membantu.

In [18]:
for min_df in [1, 2, 3]:
    try:
        vec = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b\w+\b",
            min_df=min_df,
            norm=None
        )
        X_tmp = vec.fit_transform(corpus)
        features_tmp = vec.get_feature_names_out()
        print(f"min_df={min_df} | shape={X_tmp.shape} | fitur={len(features_tmp)}")
        print("fitur:", list(features_tmp))
        print()
    except ValueError as e:
        print(f"min_df={min_df} gagal: {e}")

min_df=1 | shape=(5, 22) | fitur=22
fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']

min_df=2 | shape=(5, 7) | fitur=7
fitur: ['analisis', 'data', 'informasi', 'learning', 'machine', 'sistem', 'untuk']

min_df=3 | shape=(5, 2) | fitur=2
fitur: ['data', 'sistem']



## 18. Eksperimen 3 — Dampak `max_df`

`max_df` membuang term yang muncul di terlalu banyak dokumen.

Contoh:

```python
max_df=0.8
```

berarti term yang muncul di lebih dari 80% dokumen akan dibuang.

### Dampak konseptual

Jika `max_df` diturunkan:

1. term yang terlalu umum dibuang,
2. bisa berfungsi seperti stopword filtering otomatis,
3. fitur menjadi lebih diskriminatif,
4. tetapi term umum yang sebenarnya penting untuk domain bisa hilang.

Pada corpus kecil, perubahan `max_df` harus hati-hati karena satu dokumen saja dapat mengubah proporsi secara besar.

In [19]:
for max_df in [1.0, 0.8, 0.6, 0.4]:
    try:
        vec = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b\w+\b",
            max_df=max_df,
            norm=None
        )
        X_tmp = vec.fit_transform(corpus)
        features_tmp = vec.get_feature_names_out()
        print(f"max_df={max_df} | shape={X_tmp.shape} | fitur={len(features_tmp)}")
        print("fitur:", list(features_tmp))
        print()
    except ValueError as e:
        print(f"max_df={max_df} gagal: {e}")

max_df=1.0 | shape=(5, 22) | fitur=22
fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']

max_df=0.8 | shape=(5, 22) | fitur=22
fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']

max_df=0.6 | shape=(5, 22) | fitur=22
fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']

max_df=0.4 | shape=(5, 20) | fitur=20
fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'digunakan', 'info

## 19. Eksperimen 4 — Dampak `use_idf`

Parameter `use_idf` menentukan apakah IDF digunakan.

- `use_idf=True`: bobot = TF × IDF.
- `use_idf=False`: bobot hanya berbasis TF, lalu bisa dinormalisasi.

### Dampak konseptual

Jika `use_idf=False`, term umum tidak diberi penalti IDF. Akibatnya, kata yang sering muncul di banyak dokumen bisa tetap dominan.

Jika `use_idf=True`, term yang umum di corpus bobotnya lebih rendah dibanding term spesifik.

In [20]:
for use_idf in [False, True]:
    vec = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        use_idf=use_idf,
        smooth_idf=True,
        norm=None
    )
    X_tmp, features_tmp, df_tmp = vectorizer_report(vec, corpus, f"use_idf={use_idf}, norm=None")


use_idf=False, norm=None
Shape matriks: (5, 22)
Jumlah fitur: 22
Contoh fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']


,alami,analisis,bahasa,berkualitas,bisnis,dalam,dan,data,digunakan,informasi,keputusan,learning,machine,membantu,membutuhkan,mining,modern,nlp,pengambilan,penting,sistem,untuk
D1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
D2,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
D3,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
D4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
D5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



use_idf=True, norm=None
Shape matriks: (5, 22)
Jumlah fitur: 22
Contoh fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']


,alami,analisis,bahasa,berkualitas,bisnis,dalam,dan,data,digunakan,informasi,keputusan,learning,machine,membantu,membutuhkan,mining,modern,nlp,pengambilan,penting,sistem,untuk
D1,0.0000,1.6931,0.0000,0.0000,0.0000,0.0000,0.0000,1.4055,0.0000,1.6931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0986,1.4055,1.6931
D2,0.0000,1.6931,0.0000,0.0000,2.0986,0.0000,2.0986,1.4055,0.0000,0.0000,0.0000,1.6931,1.6931,0.0000,0.0000,2.0986,0.0000,0.0000,0.0000,0.0000,0.0000,1.6931
D3,2.0986,0.0000,2.0986,0.0000,0.0000,2.0986,0.0000,0.0000,2.0986,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0986,2.0986,0.0000,0.0000,1.4055,0.0000
D4,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.6931,2.0986,0.0000,0.0000,2.0986,0.0000,0.0000,0.0000,0.0000,2.0986,0.0000,1.4055,0.0000
D5,0.0000,0.0000,0.0000,2.0986,0.0000,0.0000,0.0000,1.4055,0.0000,0.0000,0.0000,1.6931,1.6931,0.0000,2.0986,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## 20. Eksperimen 5 — Dampak `smooth_idf`

`smooth_idf=True` menggunakan formula:

\[
IDF(t) = \log \left(\frac{1+N}{1+DF(t)}\right)+1
\]

`smooth_idf=False` dalam scikit-learn menggunakan formula:

\[
IDF(t) = \log \left(\frac{N}{DF(t)}\right)+1
\]

Perhatikan bahwa scikit-learn tetap menambahkan `+1` meskipun smoothing dimatikan.

### Dampak konseptual

`smooth_idf=True` membuat bobot IDF lebih stabil, khususnya pada corpus kecil atau term yang sangat jarang.

In [21]:
for smooth in [False, True]:
    vec = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        use_idf=True,
        smooth_idf=smooth,
        norm=None
    )
    X_tmp = vec.fit_transform(corpus)
    idf_values = pd.DataFrame({
        "term": vec.get_feature_names_out(),
        "idf": vec.idf_
    }).sort_values("idf", ascending=False)
    show_df(idf_values, f"IDF scikit-learn dengan smooth_idf={smooth}")


IDF scikit-learn dengan smooth_idf=False


,term,idf
0,alami,2.6094
2,bahasa,2.6094
4,bisnis,2.6094
3,berkualitas,2.6094
5,dalam,2.6094
6,dan,2.6094
18,pengambilan,2.6094
8,digunakan,2.6094
13,membantu,2.6094
10,keputusan,2.6094



IDF scikit-learn dengan smooth_idf=True


,term,idf
0,alami,2.0986
2,bahasa,2.0986
4,bisnis,2.0986
3,berkualitas,2.0986
5,dalam,2.0986
6,dan,2.0986
18,pengambilan,2.0986
8,digunakan,2.0986
13,membantu,2.0986
10,keputusan,2.0986


## 21. Eksperimen 6 — Dampak `sublinear_tf`

`sublinear_tf=True` mengganti TF mentah menjadi:

\[
TF = 1 + \log(tf)
\]

untuk term yang muncul.

### Dampak konseptual

Jika sebuah term muncul 10 kali, raw TF memberi bobot 10. Log TF memberi bobot:

\[
1 + \log(10) \approx 3.3026
\]

Artinya, term yang sering muncul tetap lebih kuat, tetapi tidak mendominasi secara linear.

Ini berguna untuk dokumen panjang atau dokumen yang mengulang kata tertentu secara berlebihan.

In [22]:
corpus_repeated = [
    "data data data data data analisis sistem",
    "data analisis bisnis",
    "sistem informasi modern",
]
ids_repeated = ["R1", "R2", "R3"]

for sublinear in [False, True]:
    vec = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        use_idf=True,
        smooth_idf=True,
        sublinear_tf=sublinear,
        norm=None
    )
    X_tmp = vec.fit_transform(corpus_repeated)
    df_tmp = pd.DataFrame(X_tmp.toarray(), columns=vec.get_feature_names_out(), index=ids_repeated)
    show_df(df_tmp, f"sublinear_tf={sublinear}, norm=None")


sublinear_tf=False, norm=None


,analisis,bisnis,data,informasi,modern,sistem
R1,1.2877,0.0000,6.4384,0.0000,0.0000,1.2877
R2,1.2877,1.6931,1.2877,0.0000,0.0000,0.0000
R3,0.0000,0.0000,0.0000,1.6931,1.6931,1.2877



sublinear_tf=True, norm=None


,analisis,bisnis,data,informasi,modern,sistem
R1,1.2877,0.0000,3.3601,0.0000,0.0000,1.2877
R2,1.2877,1.6931,1.2877,0.0000,0.0000,0.0000
R3,0.0000,0.0000,0.0000,1.6931,1.6931,1.2877


## 22. Eksperimen 7 — Dampak `binary`

`binary=True` membuat count menjadi 1 jika term muncul, tanpa memperhatikan jumlah kemunculan.

### Dampak konseptual

`binary=True` cocok jika yang penting adalah keberadaan term, bukan frekuensinya.

Contoh aplikasi:

- keyword detection,
- spam filtering sederhana,
- rule-like classification,
- fitur presence/absence untuk model tertentu.

Namun, binary representation kehilangan informasi intensitas kemunculan term.

In [23]:
for binary in [False, True]:
    vec = CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        binary=binary
    )
    X_tmp = vec.fit_transform(corpus_repeated)
    df_tmp = pd.DataFrame(X_tmp.toarray(), columns=vec.get_feature_names_out(), index=ids_repeated)
    show_df(df_tmp, f"CountVectorizer binary={binary}")


CountVectorizer binary=False


,analisis,bisnis,data,informasi,modern,sistem
R1,1,0,5,0,0,1
R2,1,1,1,0,0,0
R3,0,0,0,1,1,1



CountVectorizer binary=True


,analisis,bisnis,data,informasi,modern,sistem
R1,1,0,1,0,0,1
R2,1,1,1,0,0,0
R3,0,0,0,1,1,1


## 23. Eksperimen 8 — Dampak `norm`

Parameter `norm` dapat bernilai:

- `None`: tidak ada normalisasi,
- `'l1'`: L1 normalization,
- `'l2'`: L2 normalization.

### Dampak konseptual

Jika `norm=None`, dokumen panjang bisa memiliki nilai total lebih besar.

Jika `norm='l2'`, setiap dokumen memiliki panjang vektor 1. Ini umum untuk cosine similarity.

Jika `norm='l1'`, jumlah nilai absolut komponen vektor menjadi 1. Ini kadang berguna jika ingin interpretasi proporsi bobot.

In [24]:
for norm in [None, "l1", "l2"]:
    vec = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        use_idf=True,
        smooth_idf=True,
        norm=norm
    )
    X_tmp = vec.fit_transform(corpus)
    arr = X_tmp.toarray()
    norms_l1 = np.sum(np.abs(arr), axis=1)
    norms_l2 = np.sqrt(np.sum(arr ** 2, axis=1))
    norm_df = pd.DataFrame({
        "dokumen": doc_ids,
        "L1_norm": norms_l1,
        "L2_norm": norms_l2,
    })
    show_df(norm_df, f"Norma Vektor setelah TfidfVectorizer(norm={norm})")


Norma Vektor setelah TfidfVectorizer(norm=None)


,dokumen,L1_norm,L2_norm
0,D1,9.9890,4.1177
1,D2,14.4739,5.1628
2,D3,13.9971,5.3292
3,D4,9.3944,4.2491
4,D5,8.9890,4.0641



Norma Vektor setelah TfidfVectorizer(norm=l1)


,dokumen,L1_norm,L2_norm
0,D1,1.0,0.4122
1,D2,1.0,0.3567
2,D3,1.0,0.3807
3,D4,1.0,0.4523
4,D5,1.0,0.4521



Norma Vektor setelah TfidfVectorizer(norm=l2)


,dokumen,L1_norm,L2_norm
0,D1,2.4259,1.0
1,D2,2.8035,1.0
2,D3,2.6265,1.0
3,D4,2.2109,1.0
4,D5,2.2118,1.0


## 24. Eksperimen 9 — Dampak `max_features`

`max_features` membatasi jumlah fitur yang dipakai.

Contoh:

```python
max_features=10
```

hanya mengambil 10 fitur teratas berdasarkan urutan internal scikit-learn setelah proses vocabulary.

### Dampak konseptual

Jika `max_features` terlalu kecil:

1. dimensi turun,
2. memori lebih hemat,
3. training lebih cepat,
4. tetapi banyak informasi hilang.

Parameter ini berguna pada corpus besar dengan vocabulary sangat besar.

In [25]:
for max_features in [None, 15, 10, 5]:
    vec = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        max_features=max_features,
        norm=None
    )
    X_tmp = vec.fit_transform(corpus)
    print(f"max_features={max_features} | shape={X_tmp.shape}")
    print("fitur:", list(vec.get_feature_names_out()))
    print()

max_features=None | shape=(5, 22)
fitur: ['alami', 'analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'membutuhkan', 'mining', 'modern', 'nlp', 'pengambilan', 'penting', 'sistem', 'untuk']

max_features=15 | shape=(5, 15)
fitur: ['analisis', 'bahasa', 'berkualitas', 'bisnis', 'dalam', 'dan', 'data', 'digunakan', 'informasi', 'keputusan', 'learning', 'machine', 'membantu', 'sistem', 'untuk']

max_features=10 | shape=(5, 10)
fitur: ['analisis', 'berkualitas', 'dalam', 'dan', 'data', 'informasi', 'learning', 'machine', 'sistem', 'untuk']

max_features=5 | shape=(5, 5)
fitur: ['analisis', 'data', 'learning', 'machine', 'sistem']



## 25. Eksperimen 10 — Dampak `token_pattern`

`token_pattern` menentukan bentuk token yang diterima.

Default scikit-learn kurang lebih menerima token dengan minimal 2 karakter alfanumerik.

Masalahnya, pada beberapa kasus NLP:

- angka seperti `5g` penting,
- token satu karakter bisa penting,
- hashtag bisa ingin dipertahankan,
- kode produk bisa penting,
- emoticon atau simbol mungkin perlu perlakuan khusus.

Contoh berikut memperlihatkan bagaimana regex token mengubah hasil vocabulary.

In [26]:
corpus_token_pattern = [
    "AI 5G dan NLP v2 penting untuk e-KYC #data",
    "Model X bekerja di jaringan 4G dan 5G",
]

patterns = {
    "default_sklearn": r"(?u)\b\w\w+\b",
    "allow_single_char": r"(?u)\b\w+\b",
    "allow_hashtag_like": r"(?u)(?:#\w+|\b\w+\b)",
}

for name, pattern in patterns.items():
    vec = CountVectorizer(lowercase=True, token_pattern=pattern)
    X_tmp = vec.fit_transform(corpus_token_pattern)
    print(f"pattern={name}")
    print("regex:", pattern)
    print("fitur:", list(vec.get_feature_names_out()))
    print()

pattern=default_sklearn
regex: (?u)\b\w\w+\b
fitur: ['4g', '5g', 'ai', 'bekerja', 'dan', 'data', 'di', 'jaringan', 'kyc', 'model', 'nlp', 'penting', 'untuk', 'v2']

pattern=allow_single_char
regex: (?u)\b\w+\b
fitur: ['4g', '5g', 'ai', 'bekerja', 'dan', 'data', 'di', 'e', 'jaringan', 'kyc', 'model', 'nlp', 'penting', 'untuk', 'v2', 'x']

pattern=allow_hashtag_like
regex: (?u)(?:#\w+|\b\w+\b)
fitur: ['#data', '4g', '5g', 'ai', 'bekerja', 'dan', 'di', 'e', 'jaringan', 'kyc', 'model', 'nlp', 'penting', 'untuk', 'v2', 'x']



## 26. Top Terms per Dokumen

Salah satu cara memahami TF-IDF adalah melihat term dengan bobot tertinggi pada setiap dokumen.

Term dengan bobot tinggi biasanya:

1. muncul dalam dokumen tersebut,
2. tidak terlalu umum di corpus,
3. cukup informatif untuk membedakan dokumen.

In [27]:
def top_terms_per_doc(vectorizer, X, doc_ids: List[str], top_k: int = 5) -> pd.DataFrame:
    features = np.array(vectorizer.get_feature_names_out())
    arr = X.toarray()
    rows = []
    for i, doc_id in enumerate(doc_ids):
        top_idx = np.argsort(arr[i])[::-1][:top_k]
        rows.append({
            "dokumen": doc_id,
            "teks": corpus[i] if i < len(corpus) else "",
            "top_terms": ", ".join([f"{features[j]} ({arr[i,j]:.4f})" for j in top_idx if arr[i,j] > 0])
        })
    return pd.DataFrame(rows)

vec = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1,2),
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True,
    norm="l2"
)
X = vec.fit_transform(corpus)
show_df(top_terms_per_doc(vec, X, doc_ids, top_k=7), "Top Terms per Dokumen dengan TF-IDF")


Top Terms per Dokumen dengan TF-IDF


,dokumen,teks,top_terms
0,D1,analisis data penting untuk sistem informasi,"untuk sistem (0.3430), penting untuk (0.3430),..."
1,D2,data mining dan machine learning untuk analisi...,"untuk analisis (0.2806), mining dan (0.2806), ..."
2,D3,bahasa alami digunakan dalam sistem NLP modern,"sistem nlp (0.2834), nlp (0.2834), modern (0.2..."
3,D4,sistem informasi membantu pengambilan keputusan,"pengambilan keputusan (0.3592), pengambilan (0..."
4,D5,machine learning membutuhkan data berkualitas,"membutuhkan (0.3676), berkualitas (0.3676), da..."


## 27. Studi Kasus Information Retrieval Sederhana

Kita akan membuat mini search engine berbasis TF-IDF.

Langkah:

1. Fit `TfidfVectorizer` pada corpus.
2. Transform query menjadi vektor dengan vocabulary yang sama.
3. Hitung cosine similarity antara query dan setiap dokumen.
4. Urutkan dokumen berdasarkan skor similarity.

Contoh query:

```text
machine learning data
```

Query ini semestinya dekat dengan dokumen yang membahas machine learning dan data.

In [28]:
def search(query: str, vectorizer, X, corpus: List[str], doc_ids: List[str], top_k: int = 5) -> pd.DataFrame:
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame({
        "rank": range(1, len(order) + 1),
        "doc_id": [doc_ids[i] for i in order],
        "score": [scores[i] for i in order],
        "document": [corpus[i] for i in order],
    })

queries = [
    "machine learning data",
    "sistem informasi keputusan",
    "bahasa alami nlp",
    "analisis bisnis data mining",
]

for q in queries:
    show_df(search(q, vec, X, corpus, doc_ids), f"Hasil Retrieval untuk Query: {q}")


Hasil Retrieval untuk Query: machine learning data


,rank,doc_id,score,document
0,1,D5,0.5696,machine learning membutuhkan data berkualitas
1,2,D2,0.4348,data mining dan machine learning untuk analisi...
2,3,D1,0.0993,analisis data penting untuk sistem informasi
3,4,D3,0.0000,bahasa alami digunakan dalam sistem NLP modern
4,5,D4,0.0000,sistem informasi membantu pengambilan keputusan



Hasil Retrieval untuk Query: sistem informasi keputusan


,rank,doc_id,score,document
0,1,D4,0.5957,sistem informasi membantu pengambilan keputusan
1,2,D1,0.3620,analisis data penting untuk sistem informasi
2,3,D3,0.0767,bahasa alami digunakan dalam sistem NLP modern
3,4,D5,0.0000,machine learning membutuhkan data berkualitas
4,5,D2,0.0000,data mining dan machine learning untuk analisi...



Hasil Retrieval untuk Query: bahasa alami nlp


,rank,doc_id,score,document
0,1,D3,0.5669,bahasa alami digunakan dalam sistem NLP modern
1,2,D5,0.0000,machine learning membutuhkan data berkualitas
2,3,D4,0.0000,sistem informasi membantu pengambilan keputusan
3,4,D2,0.0000,data mining dan machine learning untuk analisi...
4,5,D1,0.0000,analisis data penting untuk sistem informasi



Hasil Retrieval untuk Query: analisis bisnis data mining


,rank,doc_id,score,document
0,1,D2,0.6336,data mining dan machine learning untuk analisi...
1,2,D1,0.1670,analisis data penting untuk sistem informasi
2,3,D5,0.0730,machine learning membutuhkan data berkualitas
3,4,D3,0.0000,bahasa alami digunakan dalam sistem NLP modern
4,5,D4,0.0000,sistem informasi membantu pengambilan keputusan


## 28. Perbandingan Konfigurasi Vectorizer untuk Retrieval

Sekarang kita bandingkan beberapa konfigurasi:

1. Unigram TF-IDF standar.
2. Unigram + bigram.
3. Unigram + bigram dengan `sublinear_tf=True`.
4. Tanpa IDF.
5. Dengan `min_df=2`.

Tujuannya bukan mencari konfigurasi terbaik secara universal, tetapi memahami bahwa parameter mengubah perilaku retrieval.

In [29]:
configs = [
    ("A: unigram TF-IDF", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1,1), use_idf=True, smooth_idf=True, norm="l2")),
    ("B: unigram+bigram TF-IDF", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1,2), use_idf=True, smooth_idf=True, norm="l2")),
    ("C: unigram+bigram sublinear", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1,2), use_idf=True, smooth_idf=True, sublinear_tf=True, norm="l2")),
    ("D: tanpa IDF", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1,1), use_idf=False, norm="l2")),
    ("E: min_df=2", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1,1), use_idf=True, smooth_idf=True, min_df=2, norm="l2")),
]

query = "machine learning data"
rows = []
for name, v in configs:
    X_cfg = v.fit_transform(corpus)
    result = search(query, v, X_cfg, corpus, doc_ids, top_k=3)
    rows.append({
        "konfigurasi": name,
        "jumlah_fitur": len(v.get_feature_names_out()),
        "top_1": result.iloc[0]["doc_id"],
        "top_1_score": result.iloc[0]["score"],
        "top_2": result.iloc[1]["doc_id"],
        "top_2_score": result.iloc[1]["score"],
        "top_3": result.iloc[2]["doc_id"],
        "top_3_score": result.iloc[2]["score"],
    })

show_df(pd.DataFrame(rows), f"Perbandingan Retrieval untuk Query: {query}")


Perbandingan Retrieval untuk Query: machine learning data


,konfigurasi,jumlah_fitur,top_1,top_1_score,top_2,top_2_score,top_3,top_3_score
0,A: unigram TF-IDF,22,D5,0.6832,D2,0.5378,D1,0.1728
1,B: unigram+bigram TF-IDF,46,D5,0.5696,D2,0.4348,D1,0.0993
2,C: unigram+bigram sublinear,46,D5,0.5696,D2,0.4348,D1,0.0993
3,D: tanpa IDF,22,D5,0.7746,D2,0.6124,D1,0.2357
4,E: min_df=2,7,D5,1.0000,D2,0.7573,D1,0.2008


## 29. Ringkasan Praktis Pemilihan Parameter

Tidak ada satu konfigurasi yang selalu terbaik. Namun, pedoman umum berikut dapat dipakai:

| Kondisi Data / Tujuan | Parameter yang Perlu Dipertimbangkan |
|---|---|
| Dataset kecil | Hindari `min_df` terlalu tinggi dan `ngram_range` terlalu besar |
| Dataset besar | Gunakan `min_df`, `max_df`, dan `max_features` untuk kontrol dimensi |
| Banyak frasa penting | Gunakan `ngram_range=(1,2)` atau `(1,3)` |
| Dokumen panjang dan banyak pengulangan | Gunakan `sublinear_tf=True` |
| Retrieval berbasis cosine similarity | Gunakan `norm='l2'` |
| Term umum terlalu dominan | Gunakan `use_idf=True`, `max_df`, atau stopword list |
| Perlu fitur presence/absence | Gunakan `binary=True` pada CountVectorizer atau konfigurasi count tertentu |
| Data media sosial | Atur `token_pattern`, hashtag, mention, emoji, dan slang normalization dari Notebook 02 |

Untuk eksperimen akademik, konfigurasi harus dijelaskan dan dibandingkan, bukan hanya dipakai.

## 30. Kesalahan Umum Mahasiswa dalam TF-IDF

### Kesalahan 1 — Mengira TF-IDF hanya satu rumus

Padahal TF dan IDF memiliki banyak variasi. Library yang berbeda bisa menghasilkan nilai berbeda.

### Kesalahan 2 — Mengabaikan normalisasi

Dua vektor TF-IDF sebelum dan sesudah L2 normalization dapat menghasilkan skala berbeda.

### Kesalahan 3 — Menggunakan `ngram_range` besar tanpa mempertimbangkan sparsity

Bigram dan trigram bagus, tetapi pada data kecil bisa membuat banyak fitur hanya muncul sekali.

### Kesalahan 4 — Menggunakan stopword removal agresif pada sentiment analysis

Kata seperti `tidak`, `bukan`, dan `belum` sangat penting.

### Kesalahan 5 — Tidak menjelaskan parameter eksperimen

Dalam laporan akademik, konfigurasi seperti `min_df`, `max_df`, `sublinear_tf`, `smooth_idf`, dan `norm` harus disebutkan.

### Kesalahan 6 — Membandingkan hasil manual dengan scikit-learn tanpa menyamakan formula

Jika manual memakai IDF klasik tetapi scikit-learn memakai smoothed IDF dan L2 normalization, hasilnya pasti berbeda.

## 31. Latihan Mahasiswa

### Latihan 1 — Manual TF-IDF

Diberikan corpus:

```text
D1: data data analisis sistem
D2: data mining sistem
D3: bahasa alami sistem
```

Hitung manual untuk term `data`:

1. TF raw pada setiap dokumen.
2. DF.
3. IDF klasik.
4. TF-IDF raw × IDF klasik.

### Latihan 2 — Variasi TF

Gunakan dokumen:

```text
data data data analisis sistem
```

Hitung untuk term `data`:

1. raw TF,
2. binary TF,
3. normalized TF,
4. log TF,
5. augmented TF.

Jelaskan perbedaan interpretasinya.

### Latihan 3 — Eksperimen `ngram_range`

Gunakan corpus minimal 10 dokumen Bahasa Indonesia. Bandingkan:

```python
ngram_range=(1,1)
ngram_range=(1,2)
ngram_range=(1,3)
```

Laporkan:

1. jumlah fitur,
2. contoh fitur,
3. sparsity matrix,
4. top terms per dokumen.

### Latihan 4 — Eksperimen `min_df` dan `max_df`

Bandingkan konfigurasi:

```python
min_df=1
min_df=2
max_df=1.0
max_df=0.8
```

Jelaskan term apa saja yang hilang dan mengapa.

### Latihan 5 — Mini Search Engine

Buat mini search engine TF-IDF dengan minimal 20 dokumen.

Untuk setiap query, tampilkan 5 dokumen paling relevan berdasarkan cosine similarity.

### Latihan 6 — Analisis Akademik

Tuliskan analisis 1–2 halaman:

> Bagaimana perubahan `ngram_range`, `min_df`, `max_df`, `sublinear_tf`, `smooth_idf`, dan `norm` memengaruhi representasi teks dan hasil retrieval?

## 32. Kesimpulan Notebook 03

Notebook ini menjelaskan bahwa representasi teks bukan sekadar menjalankan `TfidfVectorizer`, tetapi melibatkan keputusan metodologis.

Poin penting:

1. Vocabulary menentukan ruang fitur.
2. N-gram menentukan seberapa besar konteks lokal ditangkap.
3. TF mengukur kepentingan term dalam dokumen.
4. IDF mengukur kekhasan term dalam corpus.
5. TF-IDF menggabungkan TF dan IDF.
6. Normalisasi memengaruhi skala dan similarity.
7. Hyperparameter vectorizer mengubah jumlah fitur, bobot, sparsity, dan hasil retrieval.
8. Semua konfigurasi harus disesuaikan dengan task, ukuran dataset, dan karakteristik bahasa.

Notebook berikutnya, **04 — Vectorizer dan Studi Kasus Terpadu**, akan menggunakan konsep dari notebook ini untuk membangun pipeline yang lebih aplikatif.